# 03 – Modelling

Choose the features and build a sklearn pipeline that prepares the data. Imputing and scaling are learned from the training data only, so the test result stays honest.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from src.config import PROCESSED, FIGURES

## Load data and target

In [ ]:
df = pd.read_csv(PROCESSED / "clean_dataset_numeric.csv")

# short_sleep is already 0/1 in this file: 1 = under 7 hours, 0 = 7 hours or more
TARGET = "short_sleep"
print(df.shape)
df[TARGET].value_counts()

## Columns left out of the model

In [ ]:
# Columns left out of X (one reason each):
# - sleep_hours_per_night: short_sleep is computed from it, so it gives away the answer
# - past_month_felt_rested: measures almost the same thing as the target
# - past_month_trouble_falling_asleep: measures almost the same thing as the target
# - past_month_trouble_staying_asleep: measures almost the same thing as the target
# - survey_weight: a population weight, not a property of the student
# - household_id: just an ID, carries no information
# - food_security_level: text copy of food_security (only in clean_dataset.csv, not in the numeric file)
DROP_COLS = [
    "sleep_hours_per_night",
    "past_month_felt_rested",
    "past_month_trouble_falling_asleep",
    "past_month_trouble_staying_asleep",
    "survey_weight",
    "household_id",
]

X = df.drop(columns=DROP_COLS + [TARGET])
y = df[TARGET]

## Feature types

In [ ]:
# Numeric: real numbers, yes/no (0/1) columns and ordered scales
NUM_COLS = [
    "age",                        # years, 18-29
    "weekly_work_hours",          # hours per week
    "kids_household",             # number of children in the household
    "family_income",              # income-to-poverty ratio, ordered 1-14
    "food_security",              # 1 high ... 4 very low (ordered)
    "activity_guideline_status",  # 1 inactive ... 3 sufficiently active (ordered)
    "anxiety_frequency",          # ordered 1-5
    "depression_frequency",       # ordered 1-5
    "serious_distress",           # 1 / 2, two values (check the codebook for which is yes)
    "sex",                        # 1 male, 2 female (two values, works like yes/no)
    "housing_cost",               # 1 yes, 0 no
    "high_bloodpressure",         # 1 yes, 0 no
    "had_asthma",                 # 1 yes, 0 no
    "had_arthritis",              # 1 yes, 0 no
    "had_diabetes",               # 1 yes, 0 no
]

# Categorical: NHIS codes where the steps are not equal
CAT_COLS = [
    "education_level",
]

# Check that every column in X is in exactly one list
missing = set(X.columns) - set(NUM_COLS) - set(CAT_COLS)
extra = (set(NUM_COLS) | set(CAT_COLS)) - set(X.columns)
assert not missing, f"Columns in X but not in any list: {missing}"
assert not extra, f"Columns in a list but not in X: {extra}"

## Preprocessing and `make_pipeline`

In [ ]:
num_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

cat_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer([
    ("num", num_pipe, NUM_COLS),
    ("cat", cat_pipe, CAT_COLS),
])


def make_pipeline(model):
    """Preprocessing + model. Each call gets its own copy of the preprocessing."""
    return Pipeline([
        ("preprocess", clone(preprocess)),
        ("model", model),
    ])

## Quick check that the pipeline runs

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipe = make_pipeline(LogisticRegression(max_iter=1000))
pipe.fit(X_train, y_train)
print(f"Test accuracy: {pipe.score(X_test, y_test):.3f}")
baseline = max(y_test.mean(), 1 - y_test.mean())
print(f"Baseline (always guess the most common class): {baseline:.3f}")